# Section 1: Configuration & Authentication

First, let's set up our configuration variables and verify authentication.

In [ ]:
# Configuration variables

PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           


print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")

In [ ]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

In [ ]:
# Check required IAM permissions
print("🔐 Checking required IAM permissions...")
print()
print("This notebook requires the following permissions:")
print()
print("   1️⃣  BigQuery:")
print("      - roles/bigquery.admin (to copy census dataset)")
print()
print("   2️⃣  Dataplex:")
print("      - roles/dataplex.admin (to create aspect types)")
print("      - roles/dataplex.catalogAdmin (to update entries with aspects)")
print()

# Try to get user email
try:
    user_email = None
    if hasattr(credentials, '_service_account_email'):
        user_email = credentials._service_account_email
    elif hasattr(credentials, 'service_account_email'):
        user_email = credentials.service_account_email
    
    if user_email:
        print(f"   Your account: {user_email}")
    else:
        print("   Your account: (Unable to detect email from credentials)")
except:
    print("   Your account: (Unable to detect email)")

print()
print("💡 To grant these permissions, run these commands in your terminal:")
print()

if user_email:
    print(f"   # Grant BigQuery Admin")
    print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
    print(f"     --member='user:{user_email}' \\")
    print(f"     --role='roles/bigquery.admin'")
    print()
    print(f"   # Grant Dataplex Admin")
    print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
    print(f"     --member='user:{user_email}' \\")
    print(f"     --role='roles/dataplex.admin'")
    print()
    print(f"   # Grant Dataplex Catalog Admin")
    print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
    print(f"     --member='user:{user_email}' \\")
    print(f"     --role='roles/dataplex.catalogAdmin'")
else:
    print(f"   # Grant BigQuery Admin")
    print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
    print(f"     --member='user:YOUR_EMAIL' \\")
    print(f"     --role='roles/bigquery.admin'")
    print()
    print(f"   # Grant Dataplex Admin")
    print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
    print(f"     --member='user:YOUR_EMAIL' \\")
    print(f"     --role='roles/dataplex.admin'")
    print()
    print(f"   # Grant Dataplex Catalog Admin")
    print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
    print(f"     --member='user:YOUR_EMAIL' \\")
    print(f"     --role='roles/dataplex.catalogAdmin'")
    print()
    print("   (Replace YOUR_EMAIL with your Google Cloud account email)")

print()
print("⏱️  After granting permissions:")
print("   - Wait 1-2 minutes for IAM propagation")
print("   - Then continue with the rest of the notebook")
print()
print("⚠️  Note: If you don't have these permissions, you'll encounter errors later.")

# Section 2: Create BigQuery Dataset and Tables


## Copy Census Bureau ACS Dataset

This section copies all tables from the public BigQuery dataset `bigquery-public-data.census_bureau_acs` into your project.

**What's included:**
- ~280 tables containing US Census Bureau American Community Survey data
- Tables include blockgroup, cbsa (Core Based Statistical Area), and census tract data
- Data spans multiple years (2007-2018) and survey periods (1-year, 3-year, 5-year)

**Process:**
1. Install required libraries
2. Create destination dataset in your project
3. List all tables from source dataset
4. Copy tables individually (more reliable than Data Transfer Service)
5. Validate completion

**Estimated time:** 10-30 minutes depending on number of tables

**Note:** We use direct table copying rather than Data Transfer Service for simpler setup without requiring service account configuration.

In [ ]:
# Install required libraries for BigQuery Data Transfer
import sys
import subprocess

# Install the packages
packages = ["google-cloud-bigquery-datatransfer", "google-cloud-bigquery"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")
print()
print("⚠️  IMPORTANT: After running this cell, please restart the kernel:")
print("   - Click 'Kernel' → 'Restart' in the menu")
print("   - Or use the restart button in the toolbar")
print("   - Then continue running from the next cell")
print()
print("   This ensures the newly installed packages are available.")

In [ ]:
# Alternative: Reload packages without kernel restart (try this first)
import sys
import importlib

# Force reload the package path
if 'google.cloud.bigquery_datatransfer' in sys.modules:
    del sys.modules['google.cloud.bigquery_datatransfer']
if 'google.cloud.bigquery' in sys.modules:
    del sys.modules['google.cloud.bigquery']

# Now import
try:
    from google.cloud import bigquery
    from google.cloud import bigquery_datatransfer
    
    # Initialize BigQuery client
    bq_client = bigquery.Client(project=PROJECT_ID)
    
    # Dataset configuration
    DATASET_ID = "census_bureau_acs"
    DATASET_LOCATION = "US"  # Must match source dataset location
    
    # Create dataset
    dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = DATASET_LOCATION
    
    dataset = bq_client.create_dataset(dataset, exists_ok=True)
    print(f"✅ Dataset created/verified: {dataset_id}")
    print(f"   Location: {DATASET_LOCATION}")
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print()
    print("Please restart the kernel and try again:")
    print("   - Click 'Kernel' → 'Restart' in the menu")
    print("   - Then run cells 1-3 again (config and auth)")
    print("   - Skip cell 6 (already installed)")
    print("   - Continue from this cell")
    raise

In [ ]:
# List all tables in source dataset to copy
# Using direct table copying instead of Data Transfer Service (simpler setup)

SOURCE_PROJECT_ID = "bigquery-public-data"
SOURCE_DATASET_ID = "census_bureau_acs"
source_dataset_ref = f"{SOURCE_PROJECT_ID}.{SOURCE_DATASET_ID}"

print(f"📋 Preparing to copy tables:")
print(f"   Source: {source_dataset_ref}")
print(f"   Destination: {dataset_id}")
print()
print("🔍 Listing tables in source dataset...")

# List all tables in the source dataset
source_tables = list(bq_client.list_tables(source_dataset_ref))
total_tables = len(source_tables)

print(f"   Found {total_tables} tables in source")
print()

# Check which tables already exist in destination
print("🔍 Checking for existing tables in destination...")
existing_tables = list(bq_client.list_tables(dataset_id))
existing_table_ids = {table.table_id for table in existing_tables}
existing_count = len(existing_table_ids)

print(f"   Found {existing_count} tables already in destination")
print()

# Filter to only tables that don't exist in destination
tables_to_copy = [table for table in source_tables 
                  if table.table_id not in existing_table_ids]
tables_to_copy_count = len(tables_to_copy)

# Report status
if existing_count > 0:
    print(f"📊 Copy Status:")
    print(f"   ✅ Already copied: {existing_count} tables")
    print(f"   📋 Remaining to copy: {tables_to_copy_count} tables")
    print()

if tables_to_copy_count == 0:
    print("✅ All tables already exist in destination - nothing to copy!")
else:
    # Show first 10 tables to be copied as preview
    print(f"📋 First 10 tables to copy:")
    for i, table in enumerate(tables_to_copy[:10]):
        print(f"   {i+1}. {table.table_id}")
    if tables_to_copy_count > 10:
        print(f"   ... and {tables_to_copy_count - 10} more")
    
    print()
    print(f"✅ Ready to copy {tables_to_copy_count} tables")

In [ ]:
# Copy all tables from source to destination
import time

# Check if there are any tables to copy
if tables_to_copy_count == 0:
    print("✅ No tables to copy - all tables already exist in destination!")
else:
    print("🚀 Starting table copy operations...")
    print(f"   Copying {tables_to_copy_count} tables (skipping {existing_count} already copied)")
    print()

    # Track progress
    copied_tables = []
    failed_tables = []

    # Copy tables in batches
    for i, source_table in enumerate(tables_to_copy):
        table_name = source_table.table_id
        source_table_ref = f"{SOURCE_PROJECT_ID}.{SOURCE_DATASET_ID}.{table_name}"
        dest_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
        
        try:
            # Create copy job
            job_config = bigquery.CopyJobConfig()
            job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
            
            copy_job = bq_client.copy_table(
                source_table_ref,
                dest_table_ref,
                job_config=job_config
            )
            
            # Wait for job to complete (with timeout)
            copy_job.result(timeout=300)  # 5 minute timeout per table
            
            copied_tables.append(table_name)
            
            # Print progress every 10 tables
            if (i + 1) % 10 == 0:
                print(f"   ✅ Copied {i + 1}/{tables_to_copy_count} tables...")
            
        except Exception as e:
            print(f"   ❌ Failed to copy {table_name}: {str(e)[:100]}")
            failed_tables.append((table_name, str(e)))
            continue

    print()
    print("=" * 60)
    print(f"✅ Copy operation completed!")
    print(f"   Successfully copied: {len(copied_tables)} tables")
    if failed_tables:
        print(f"   Failed: {len(failed_tables)} tables")
    print(f"   Total in destination: {existing_count + len(copied_tables)} tables")
    print("=" * 60)

In [ ]:
# Show any failed tables
if failed_tables:
    print()
    print("⚠️  Failed Tables:")
    for table_name, error in failed_tables[:10]:  # Show first 10
        print(f"   - {table_name}: {error[:80]}...")
    if len(failed_tables) > 10:
        print(f"   ... and {len(failed_tables) - 10} more failures")
else:
    print()
    print("✅ All tables copied successfully!")

In [ ]:
# Validate dataset copy
print("🔍 Validating copied dataset...")
print()

# List tables in destination dataset
dest_tables = list(bq_client.list_tables(dataset_id))
table_count = len(dest_tables)

print(f"✅ Destination dataset: {dataset_id}")
print(f"   Tables in destination: {table_count}")
print(f"   Expected: {total_tables}")
print()

# Show first 10 tables as sample
print("📋 Sample tables in destination:")
for i, table in enumerate(dest_tables[:10]):
    print(f"   {i+1}. {table.table_id}")

if table_count > 10:
    print(f"   ... and {table_count - 10} more tables")

print()

# Verify we have the expected number of tables
if table_count >= total_tables - 5:  # Allow small variance
    print(f"✅ Checkpoint passed: Dataset copied successfully!")
    print(f"   Expected {total_tables} tables, found {table_count}")
else:
    print(f"⚠️  Warning: Expected {total_tables} tables but found {table_count}")
    print(f"   Some tables may not have been copied")

print()
print(f"🔗 View in Console:")
print(f"   https://console.cloud.google.com/bigquery?project={PROJECT_ID}&ws=!1m5!1m4!4m3!1s{PROJECT_ID}!2s{DATASET_ID}!3sblockgroup_2018_5yr")

In [ ]:
# View dataset in BigQuery Console
print()
print("🔗 Quick Links:")
print(f"   Dataset: https://console.cloud.google.com/bigquery?project={PROJECT_ID}&d={DATASET_ID}&p={PROJECT_ID}&page=dataset")
print(f"   Sample table: https://console.cloud.google.com/bigquery?project={PROJECT_ID}&ws=!1m5!1m4!4m3!1s{PROJECT_ID}!2s{DATASET_ID}!3sblockgroup_2018_5yr")
print()
print("✅ Section 2 complete! Census Bureau ACS dataset is now available in your project.")

# Section 3: Create Dataplex Aspect Type

This section creates a custom Dataplex Aspect Type to capture census-specific metadata for the ACS dataset tables.

## Create Census Survey Metadata Aspect Type

An **Aspect Type** in Dataplex defines a reusable metadata template that can be attached to data assets. This aspect type will capture:

- **Survey Vintage**: The year the census data represents (e.g., 2022)
- **Estimate Period**: Whether it's a 1-year or 5-year estimate
- **Experimental Data**: Flag to indicate if the Census Bureau labeled this as experimental

This metadata enriches the data catalog and makes it easier to understand the nature and quality of each dataset.

In [ ]:
# Install Dataplex library
import sys
import subprocess

print("📦 Installing Dataplex library...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-cloud-dataplex"])

print("✅ Library installed")
print()
print("⚠️  IMPORTANT: After running this cell, please restart the kernel:")
print("   - Click 'Kernel' → 'Restart' in the menu")
print("   - Or use the restart button in the toolbar")
print("   - Then continue running from the next cell")
print()
print("   This ensures the newly installed package is available.")

In [ ]:
# Alternative: Reload packages without kernel restart (try this first)
import sys

# Force reload the package path
if 'google.cloud.dataplex' in sys.modules:
    del sys.modules['google.cloud.dataplex']
if 'google.cloud.dataplex_v1' in sys.modules:
    del sys.modules['google.cloud.dataplex_v1']

# Now import
try:
    from google.cloud import dataplex_v1
    
    # Initialize Catalog client (for aspect types)
    catalog_client = dataplex_v1.CatalogServiceClient()
    
    print("✅ Dataplex Catalog client initialized")
    print(f"   Project: {PROJECT_ID}")
    print(f"   Region: {REGION}")
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print()
    print("Please restart the kernel and try again:")
    print("   - Click 'Kernel' → 'Restart' in the menu")
    print("   - Then run cells 1-3 again (config and auth)")
    print("   - Skip cell 15 (already installed)")
    print("   - Continue from this cell")
    raise

In [ ]:
# Define the Aspect Type fields for Census Survey Metadata
print("🔧 Defining Census Survey Metadata Aspect Type...")
print()

# IMPORTANT: Use 'us' location to match BigQuery dataset location
# The BigQuery dataset is in US multi-region, so its Dataplex catalog entries
# are in the 'us' location. Aspect types must be in the same location as entries.
CATALOG_LOCATION = "us"
parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"
aspect_type_id = "census-survey-metadata"

print(f"   Creating aspect type in location: {CATALOG_LOCATION}")
print(f"   (Matches BigQuery US multi-region dataset location)")
print()

# Define each field as a MetadataTemplate
# Field 1: survey_vintage (int, required)
survey_vintage_field = dataplex_v1.AspectType.MetadataTemplate(
    name="survey_vintage",
    type="int",  # Valid types: datetime, double, bool, int, string, record, map, array, enum
    index=1,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        description="The year the data represents (e.g., 2022).",
        display_name="Survey Year (Vintage)"
    ),
    constraints=dataplex_v1.AspectType.MetadataTemplate.Constraints(
        required=True
    )
)

# Field 2: estimate_period (enum, required)
estimate_period_field = dataplex_v1.AspectType.MetadataTemplate(
    name="estimate_period",
    type="enum",
    index=2,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        description="The duration of the survey collection.",
        display_name="Estimate Period"
    ),
    constraints=dataplex_v1.AspectType.MetadataTemplate.Constraints(
        required=True
    ),
    enum_values=[
        dataplex_v1.AspectType.MetadataTemplate.EnumValue(
            index=1,
            name="1_year"
        ),
        dataplex_v1.AspectType.MetadataTemplate.EnumValue(
            index=2,
            name="5_year"
        )
    ]
)

# Field 3: is_experimental (bool)
is_experimental_field = dataplex_v1.AspectType.MetadataTemplate(
    name="is_experimental",
    type="bool",  # Valid types: datetime, double, bool, int, string, record, map, array, enum
    index=3,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        description="Set to true if the Census Bureau labels this as experimental (e.g., 2020 1-year).",
        display_name="Experimental Data"
    )
)

# Create the aspect type with all fields
aspect_type = dataplex_v1.AspectType(
    description="Captures the vintage and estimation type for US Census Bureau ACS datasets.",
    metadata_template=dataplex_v1.AspectType.MetadataTemplate(
        name="census_survey_metadata_template",
        type="record",
        record_fields=[survey_vintage_field, estimate_period_field, is_experimental_field]
    )
)

print(f"📋 Aspect Type Configuration:")
print(f"   ID: {aspect_type_id}")
print(f"   Description: {aspect_type.description}")
print(f"   Location: {parent}")
print(f"   Fields:")
print(f"     1. survey_vintage (int, required)")
print(f"     2. estimate_period (enum: 1_year, 5_year, required)")
print(f"     3. is_experimental (bool)")

In [ ]:
# Create the Aspect Type
print()
print("🚀 Creating aspect type in Dataplex...")

# Check if catalog_client exists, if not initialize it
try:
    catalog_client
except NameError:
    print("   ℹ️  Initializing Dataplex Catalog client...")
    from google.cloud import dataplex_v1
    catalog_client = dataplex_v1.CatalogServiceClient()
    print("   ✅ Client initialized")

try:
    # Create the aspect type using the catalog client
    create_operation = catalog_client.create_aspect_type(
        parent=parent,
        aspect_type=aspect_type,
        aspect_type_id=aspect_type_id
    )
    
    # Wait for the operation to complete
    print("   ⏳ Waiting for creation to complete...")
    result = create_operation.result(timeout=300)  # 5 minute timeout
    
    print()
    print("=" * 60)
    print("✅ Aspect Type created successfully!")
    print("=" * 60)
    print(f"   Resource name: {result.name}")
    print(f"   Description: {result.description}")
    print(f"   UID: {result.uid}")
    
except Exception as e:
    error_message = str(e)
    
    # Check if aspect type already exists
    if "ALREADY_EXISTS" in error_message or "already exists" in error_message:
        print()
        print("⚠️  Aspect Type already exists")
        print(f"   Name: projects/{PROJECT_ID}/locations/{REGION}/aspectTypes/{aspect_type_id}")
        print()
        print("   To update the aspect type, you'll need to delete it first:")
        print(f"   gcloud dataplex aspect-types delete {aspect_type_id} \\")
        print(f"     --project={PROJECT_ID} \\")
        print(f"     --location={REGION}")
        print()
        print("   Or continue - the existing aspect type will be used.")
    else:
        print(f"❌ Failed to create aspect type: {e}")
        raise

In [ ]:
# Validate the Aspect Type
print()
print("🔍 Validating Aspect Type...")

# Check if catalog_client exists, if not initialize it
try:
    catalog_client
except NameError:
    print("   ℹ️  Initializing Dataplex Catalog client...")
    from google.cloud import dataplex_v1
    catalog_client = dataplex_v1.CatalogServiceClient()
    print("   ✅ Client initialized")

try:
    # Construct the full resource name
    aspect_type_name = f"projects/{PROJECT_ID}/locations/{REGION}/aspectTypes/{aspect_type_id}"
    
    # Get the aspect type to verify it exists
    retrieved_aspect_type = catalog_client.get_aspect_type(name=aspect_type_name)
    
    print()
    print("✅ Aspect Type verified:")
    print(f"   Resource name: {retrieved_aspect_type.name}")
    print(f"   Description: {retrieved_aspect_type.description}")
    print(f"   UID: {retrieved_aspect_type.uid}")
    print(f"   Created: {retrieved_aspect_type.create_time}")
    print()
    
    # Display the fields
    print("📋 Metadata Template Fields:")
    template = retrieved_aspect_type.metadata_template
    if template.record_fields:
        for field in sorted(template.record_fields, key=lambda f: f.index):
            required = " (required)" if field.constraints and field.constraints.required else ""
            print(f"   {field.index}. {field.name} ({field.type}){required}")
            if field.annotations:
                if field.annotations.display_name:
                    print(f"      Display: {field.annotations.display_name}")
                if field.annotations.description:
                    print(f"      Description: {field.annotations.description}")
            
            if field.enum_values:
                enum_names = [v.name for v in field.enum_values]
                print(f"      Allowed values: {', '.join(enum_names)}")
    
    print()
    print("✅ Checkpoint passed: Aspect Type created and validated!")
    
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise

## Create Public Data Governance Aspect Type

This aspect type captures governance metadata for external public datasets, including:

- **Source Agency**: The government agency or organization that published the data
- **License Type**: The data license (usually Public Domain for US Government data)
- **Last Cataloged**: When this dataset was last cataloged or verified

In [ ]:
# Define the Aspect Type fields for Public Data Governance
print("🔧 Defining Public Data Governance Aspect Type...")
print()

# Define the aspect type ID
governance_aspect_type_id = "data-governance-public"

# Field 1: source_agency (string)
source_agency_field = dataplex_v1.AspectType.MetadataTemplate(
    name="source_agency",
    type="string",
    index=1,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        display_name="Source Agency"
    )
)

# Field 2: data_license (string)
data_license_field = dataplex_v1.AspectType.MetadataTemplate(
    name="data_license",
    type="string",
    index=2,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        display_name="License Type",
        description="Usually Public Domain for US Government data."
    )
)

# Field 3: last_cataloged_date (datetime)
last_cataloged_field = dataplex_v1.AspectType.MetadataTemplate(
    name="last_cataloged_date",
    type="datetime",
    index=3,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        display_name="Last Cataloged"
    )
)

# Create the aspect type with all fields
governance_aspect_type = dataplex_v1.AspectType(
    description="Governance metadata for external public datasets.",
    metadata_template=dataplex_v1.AspectType.MetadataTemplate(
        name="data_governance_public_template",
        type="record",
        record_fields=[source_agency_field, data_license_field, last_cataloged_field]
    )
)

print(f"📋 Aspect Type Configuration:")
print(f"   ID: {governance_aspect_type_id}")
print(f"   Description: {governance_aspect_type.description}")
print(f"   Location: {parent}")
print(f"   Fields:")
print(f"     1. source_agency (string)")
print(f"     2. data_license (string)")
print(f"     3. last_cataloged_date (datetime)")

In [ ]:
# Create the Public Data Governance Aspect Type
print()
print("🚀 Creating Public Data Governance aspect type in Dataplex...")

# Check if catalog_client exists, if not initialize it
try:
    catalog_client
except NameError:
    print("   ℹ️  Initializing Dataplex Catalog client...")
    from google.cloud import dataplex_v1
    catalog_client = dataplex_v1.CatalogServiceClient()
    print("   ✅ Client initialized")

try:
    # Create the aspect type using the catalog client
    create_operation = catalog_client.create_aspect_type(
        parent=parent,
        aspect_type=governance_aspect_type,
        aspect_type_id=governance_aspect_type_id
    )
    
    # Wait for the operation to complete
    print("   ⏳ Waiting for creation to complete...")
    result = create_operation.result(timeout=300)  # 5 minute timeout
    
    print()
    print("=" * 60)
    print("✅ Public Data Governance Aspect Type created successfully!")
    print("=" * 60)
    print(f"   Resource name: {result.name}")
    print(f"   Description: {result.description}")
    print(f"   UID: {result.uid}")
    
except Exception as e:
    error_message = str(e)
    
    # Check if aspect type already exists
    if "ALREADY_EXISTS" in error_message or "already exists" in error_message:
        print()
        print("⚠️  Aspect Type already exists")
        print(f"   Name: projects/{PROJECT_ID}/locations/{REGION}/aspectTypes/{governance_aspect_type_id}")
        print()
        print("   To update the aspect type, you'll need to delete it first:")
        print(f"   gcloud dataplex aspect-types delete {governance_aspect_type_id} \\")
        print(f"     --project={PROJECT_ID} \\")
        print(f"     --location={REGION}")
        print()
        print("   Or continue - the existing aspect type will be used.")
    else:
        print(f"❌ Failed to create aspect type: {e}")
        raise

In [ ]:
# Validate the Public Data Governance Aspect Type
print()
print("🔍 Validating Public Data Governance Aspect Type...")

# Check if catalog_client exists, if not initialize it
try:
    catalog_client
except NameError:
    print("   ℹ️  Initializing Dataplex Catalog client...")
    from google.cloud import dataplex_v1
    catalog_client = dataplex_v1.CatalogServiceClient()
    print("   ✅ Client initialized")

try:
    # Construct the full resource name
    governance_aspect_type_name = f"projects/{PROJECT_ID}/locations/{REGION}/aspectTypes/{governance_aspect_type_id}"
    
    # Get the aspect type to verify it exists
    retrieved_governance_aspect_type = catalog_client.get_aspect_type(name=governance_aspect_type_name)
    
    print()
    print("✅ Public Data Governance Aspect Type verified:")
    print(f"   Resource name: {retrieved_governance_aspect_type.name}")
    print(f"   Description: {retrieved_governance_aspect_type.description}")
    print(f"   UID: {retrieved_governance_aspect_type.uid}")
    print(f"   Created: {retrieved_governance_aspect_type.create_time}")
    print()
    
    # Display the fields
    print("📋 Metadata Template Fields:")
    template = retrieved_governance_aspect_type.metadata_template
    if template.record_fields:
        for field in sorted(template.record_fields, key=lambda f: f.index):
            required = " (required)" if field.constraints and field.constraints.required else ""
            print(f"   {field.index}. {field.name} ({field.type}){required}")
            if field.annotations:
                if field.annotations.display_name:
                    print(f"      Display: {field.annotations.display_name}")
                if field.annotations.description:
                    print(f"      Description: {field.annotations.description}")
    
    print()
    print("✅ Checkpoint passed: Public Data Governance Aspect Type created and validated!")
    
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise

# Section 4: Apply Aspects to BigQuery Tables

Now that we have created both aspect types (Census Survey Metadata and Public Data Governance), we'll apply them to all the BigQuery tables in our census dataset.

**What this section does:**
- Discovers the BigQuery entry group in Dataplex Catalog
- Parses table names to extract vintage (year) and estimate period (1-year or 5-year)
- Applies both aspect types to each table with appropriate metadata values
- Validates that aspects were successfully attached

**Why this matters:**
- Enriches data catalog with searchable, structured metadata
- Makes it easy to find tables by year, estimate period, or governance attributes
- Provides consistent metadata across all census tables
- Enables better data discovery and governance

In [ ]:
# Helper function to parse table names for survey metadata
import re

def get_survey_metadata(table_name):
    """
    Parses BigQuery table name to extract census survey metadata.
    
    Examples:
        blockgroup_2018_5yr -> (2018, "5_year")
        cbsa_2010_1yr -> (2010, "1_year")
        census_tract_2015_5yr -> (2015, "5_year")
    
    Args:
        table_name: BigQuery table name (e.g., "blockgroup_2018_5yr")
        
    Returns:
        tuple: (vintage as int, period as enum string)
    """
    # Extract year (4 consecutive digits)
    year_match = re.search(r"(\d{4})", table_name)
    vintage = int(year_match.group(1)) if year_match else 2018
    
    # Extract period (1yr or 5yr)
    period_match = re.search(r"([15])yr", table_name)
    if period_match:
        period_num = period_match.group(1)
        period = "1_year" if period_num == "1" else "5_year"
    else:
        period = "5_year"  # Default to 5-year if not specified
    
    return vintage, period

# Test the function with a few examples
print("🧪 Testing table name parser:")
test_tables = ["blockgroup_2018_5yr", "cbsa_2010_1yr", "census_tract_2015_5yr"]
for table in test_tables:
    vintage, period = get_survey_metadata(table)
    print(f"   {table} -> vintage={vintage}, period={period}")

print()
print("✅ Helper function ready")

In [85]:
# Diagnostic: List entries in @bigquery entry group to find the correct format
print("🔍 Diagnostic: Listing entries in @bigquery entry group...")
print()

try:
    # List entries in the @bigquery entry group
    entry_group_path = f"projects/{PROJECT_ID}/locations/us/entryGroups/@bigquery"
    
    print(f"   Entry group: {entry_group_path}")
    print(f"   Listing entries...")
    print()
    
    # List entries (get first page to see the pattern)
    entries = catalog_client.list_entries(parent=entry_group_path)
    
    found_sample = False
    sample_table = "blockgroup_2018_5yr"
    
    for i, entry in enumerate(entries):
        # Check if this is our census dataset
        if DATASET_ID in entry.name or sample_table in entry.name:
            print(f"   ✅ Found census entry #{i+1}:")
            print(f"      Entry name: {entry.name}")
            print(f"      Entry type: {entry.entry_type}")
            
            # Extract entry ID
            entry_id = entry.name.split("/entries/")[-1]
            print(f"      Entry ID: {entry_id}")
            
            if sample_table in entry.name:
                found_sample = True
                print(f"      >>> This is our sample table!")
                
                # Check aspects
                if entry.aspects:
                    print(f"      Current aspects: {len(entry.aspects)}")
                    for aspect_key in entry.aspects.keys():
                        print(f"         - {aspect_key}")
                else:
                    print(f"      Current aspects: None")
            
            print()
            
            # Stop after finding a few census entries
            if found_sample or i >= 5:
                break
    
    if not found_sample:
        print("   ⚠️  Didn't find our sample table in first 10 entries")
        print("   But we can see the entry format. Continue to apply aspects...")
    
    print()
    print("✅ Now we know the correct entry path format!")
    print("   Continue to next cells to create aspect types and apply them.")
    
except Exception as e:
    error_msg = str(e)
    print(f"   ❌ Failed to list entries: {error_msg[:300]}")
    print()
    
    if "404" in error_msg or "not found" in error_msg.lower():
        print("   The @bigquery entry group doesn't exist in 'us' location!")
        print()
        print("   This means BigQuery tables are NOT cataloged in Dataplex yet.")
        print("   They may be visible in the UI but not available as catalog entries.")
        print()
        print("   Possible solutions:")
        print("   1. Wait for automatic cataloging (can take time)")
        print("   2. Check if cataloging needs to be enabled")
        print("   3. Use a different approach to add metadata")
    elif "403" in error_msg or "Permission denied" in error_msg:
        print("   Permission denied to list entries")
        print("   Need: dataplex.entries.list permission")
    else:
        print("   Unexpected error")
    
    raise

🔍 Diagnostic: Listing entries in @bigquery entry group...

   Entry group: projects/johnswain-1200-20260303091357/locations/us/entryGroups/@bigquery
   Listing entries...

   ✅ Found census entry #1:
      Entry name: projects/johnswain-1200-20260303091357/locations/us/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/johnswain-1200-20260303091357/datasets/census_bureau_acs/tables/blockgroup_2012_5yr
      Entry type: projects/655216118709/locations/global/entryTypes/bigquery-table
      Entry ID: bigquery.googleapis.com/projects/johnswain-1200-20260303091357/datasets/census_bureau_acs/tables/blockgroup_2012_5yr

   ✅ Found census entry #3:
      Entry name: projects/johnswain-1200-20260303091357/locations/us/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/johnswain-1200-20260303091357/datasets/census_bureau_acs/tables/puma_2010_5yr
      Entry type: projects/655216118709/locations/global/entryTypes/bigquery-table
      Entry ID: bigquery.googleapis.com/proj

In [86]:
# Verify aspect types exist in correct location
print("✅ Verifying aspect types are in correct location...")
print()

# The aspect types should have been created in 'us' location in Section 3
CATALOG_LOCATION = "us"
catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"

survey_aspect_type_name = f"{catalog_parent}/aspectTypes/census-survey-metadata"
gov_aspect_type_name = f"{catalog_parent}/aspectTypes/data-governance-public"

print(f"   Catalog location: {CATALOG_LOCATION}")
print(f"   Aspect types:")
print(f"      - {survey_aspect_type_name}")
print(f"      - {gov_aspect_type_name}")
print()

# Verify they exist
try:
    survey_type = catalog_client.get_aspect_type(name=survey_aspect_type_name)
    gov_type = catalog_client.get_aspect_type(name=gov_aspect_type_name)
    
    print("✅ Both aspect types verified in 'us' location!")
    print()
    print("   Ready to apply aspects to BigQuery entries.")
    
except Exception as e:
    print(f"❌ Aspect types not found: {str(e)[:200]}")
    print()
    print("   Please run Section 3 cells to create the aspect types.")
    print("   Make sure they are created in 'us' location (not us-central1).")
    raise

🔧 Recreating aspect types in 'us' location...

   Creating aspect types in: us

   1️⃣  Creating Census Survey Metadata aspect type...
      ✅ Created: projects/johnswain-1200-20260303091357/locations/us/aspectTypes/census-survey-metadata
   2️⃣  Creating Public Data Governance aspect type...
      ✅ Created: projects/johnswain-1200-20260303091357/locations/us/aspectTypes/data-governance-public

✅ Aspect types ready in correct location!
   Newly created: 2
      - census-survey-metadata
      - data-governance-public

   Now both aspect types and BigQuery entries are in 'us' location!
   Continue to the next cell to apply aspects to the tables.


In [87]:
# Use the correct location for BigQuery entries
print("📍 Setting up correct location for BigQuery entries...")
print()

# BigQuery datasets in US multi-region are cataloged in 'us' location
CATALOG_LOCATION = "us"
catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"
BQ_ENTRY_GROUP = "@bigquery"

print(f"   Catalog location: {CATALOG_LOCATION}")
print(f"   Entry group: {BQ_ENTRY_GROUP}")
print(f"   Parent path: {catalog_parent}")
print()
print("✅ Location configured correctly for US multi-region BigQuery dataset")

📍 Setting up correct location for BigQuery entries...

   Catalog location: us
   Entry group: @bigquery
   Parent path: projects/johnswain-1200-20260303091357/locations/us

✅ Location configured correctly for US multi-region BigQuery dataset


In [88]:
# Apply both aspects to all BigQuery tables
import datetime
from google.cloud import dataplex_v1
from google.protobuf import struct_pb2
from google.protobuf.field_mask_pb2 import FieldMask

print("🚀 Applying aspects to all census tables...")
print()

# Verify required variables exist from previous sections
try:
    DATASET_ID
    CATALOG_LOCATION
    catalog_parent
    print(f"   Using dataset: {DATASET_ID}")
    print(f"   Catalog location: {CATALOG_LOCATION}")
except NameError as e:
    print(f"   ❌ Required variable not found: {e}")
    print("   Please run previous Section 4 cells first.")
    raise

# Check if bq_client exists, if not initialize it
try:
    bq_client
except NameError:
    from google.cloud import bigquery
    bq_client = bigquery.Client(project=PROJECT_ID)

# List all tables in the dataset
dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"
tables = list(bq_client.list_tables(dataset_ref))
total_tables = len(tables)

print(f"📋 Found {total_tables} tables to process")
print()

# Aspect type paths - use CATALOG_LOCATION
SURVEY_ASPECT_TYPE = f"{catalog_parent}/aspectTypes/census-survey-metadata"
GOV_ASPECT_TYPE = f"{catalog_parent}/aspectTypes/data-governance-public"

# Current timestamp for governance metadata
now = datetime.datetime.utcnow()
timestamp_str = now.isoformat() + "Z"

# Track progress
success_count = 0
error_count = 0
errors = []

# Fail early threshold
MAX_CONSECUTIVE_ERRORS = 3
consecutive_errors = 0

print("⏳ Processing tables...")
for i, table in enumerate(tables):
    table_id = table.table_id
    
    # Parse table name for metadata
    vintage, period = get_survey_metadata(table_id)
    
    # Construct the Dataplex entry name using the correct format
    # Format: bigquery.googleapis.com/projects/{project}/datasets/{dataset}/tables/{table}
    entry_id = f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
    entry_name = f"{catalog_parent}/entryGroups/{BQ_ENTRY_GROUP}/entries/{entry_id}"
    
    try:
        # Aspect keys must be in format: {project}.{location}.{aspect_type_id}
        survey_aspect_key = f"{PROJECT_ID}.{CATALOG_LOCATION}.census-survey-metadata"
        gov_aspect_key = f"{PROJECT_ID}.{CATALOG_LOCATION}.data-governance-public"
        
        # Create Entry object with both aspects
        entry = dataplex_v1.Entry(
            name=entry_name,
            aspects={
                # Survey metadata aspect
                survey_aspect_key: dataplex_v1.Aspect(
                    aspect_type=SURVEY_ASPECT_TYPE,
                    data=struct_pb2.Struct(
                        fields={
                            "survey_vintage": struct_pb2.Value(number_value=vintage),
                            "estimate_period": struct_pb2.Value(string_value=period),
                            "is_experimental": struct_pb2.Value(bool_value=False)
                        }
                    )
                ),
                # Governance metadata aspect
                gov_aspect_key: dataplex_v1.Aspect(
                    aspect_type=GOV_ASPECT_TYPE,
                    data=struct_pb2.Struct(
                        fields={
                            "source_agency": struct_pb2.Value(string_value="U.S. Census Bureau"),
                            "data_license": struct_pb2.Value(string_value="Public Domain (U.S. Government Work)"),
                            "last_cataloged_date": struct_pb2.Value(string_value=timestamp_str)
                        }
                    )
                )
            }
        )
        
        # Update mask to specify we're updating aspects
        update_mask = FieldMask(paths=["aspects"])
        
        # Update the entry with the aspects
        catalog_client.update_entry(entry=entry, update_mask=update_mask)
        success_count += 1
        consecutive_errors = 0  # Reset on success
        
    except Exception as e:
        error_count += 1
        consecutive_errors += 1
        error_msg = str(e)
        errors.append((table_id, error_msg[:100]))
        
        # Check if this is a fatal error that should stop processing
        if "403" in error_msg or "Permission denied" in error_msg:
            print()
            print(f"❌ Permission denied error on table {i+1}/{total_tables}")
            print(f"   Table: {table_id}")
            print(f"   Error: {error_msg[:200]}")
            print()
            print("   Required permissions: roles/dataplex.catalogAdmin")
            print("   After granting, wait 1-2 minutes for propagation, then re-run this cell.")
            raise
        
        # Stop after too many consecutive errors
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            print()
            print(f"❌ Stopping after {consecutive_errors} consecutive errors")
            print(f"   Processed {i+1}/{total_tables} tables before stopping")
            print()
            print("   Recent errors:")
            for err_table, err_msg in errors[-3:]:
                print(f"   - {err_table}: {err_msg}")
            raise Exception(f"Too many consecutive errors ({consecutive_errors})")
    
    # Show progress every 25 tables
    if (i + 1) % 25 == 0:
        print(f"   Processed {i + 1}/{total_tables} tables... ({success_count} successful)")

print()
print("=" * 60)
print("✅ Aspect application completed!")
print("=" * 60)
print(f"   Tables processed: {total_tables}")
print(f"   Successfully updated: {success_count}")
print(f"   Errors: {error_count}")

if error_count > 0:
    print()
    print(f"⚠️  Errors encountered ({error_count}):")
    for table_id, error in errors[:5]:
        print(f"   - {table_id}: {error}")
    if len(errors) > 5:
        print(f"   ... and {len(errors) - 5} more errors")
else:
    print()
    print("✅ All aspects applied successfully!")

🚀 Applying aspects to all census tables...

   Using dataset: census_bureau_acs
   Catalog location: us
📋 Found 278 tables to process

⏳ Processing tables...
   Processed 25/278 tables... (25 successful)
   Processed 50/278 tables... (50 successful)
   Processed 75/278 tables... (75 successful)
   Processed 100/278 tables... (100 successful)
   Processed 125/278 tables... (125 successful)
   Processed 150/278 tables... (150 successful)
   Processed 175/278 tables... (175 successful)
   Processed 200/278 tables... (200 successful)
   Processed 225/278 tables... (225 successful)
   Processed 250/278 tables... (250 successful)
   Processed 275/278 tables... (275 successful)

✅ Aspect application completed!
   Tables processed: 278
   Successfully updated: 278
   Errors: 0

✅ All aspects applied successfully!


In [ ]:
# Validate applied aspects by reading them back
print("🔍 Validating applied aspects...")
print()

# Pick a sample table to validate
sample_table = "blockgroup_2018_5yr"

# Use the correct entry ID format
entry_id = f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{sample_table}"
entry_name = f"{catalog_parent}/entryGroups/{BQ_ENTRY_GROUP}/entries/{entry_id}"

print(f"📋 Checking aspects on sample table: {sample_table}")
print(f"   Entry: {entry_name}")
print()

# Check both aspects that should have been applied
aspect_ids = ["census-survey-metadata", "data-governance-public"]
found_aspects = 0

try:
    for aspect_id in aspect_ids:
        # Aspect key format: {project}.{location}.{aspect_type_id}
        aspect_key = f"{PROJECT_ID}.{CATALOG_LOCATION}.{aspect_id}"
        aspect_full_name = f"{entry_name}/aspects/{aspect_key}"
        
        try:
            # Try to get the specific aspect
            aspect = catalog_client.get_aspect(name=aspect_full_name)
            
            aspect_type_name = aspect.aspect_type.split("/")[-1]
            
            print(f"✅ Found aspect: {aspect_id}")
            print(f"   Type: {aspect_type_name}")
            print(f"   Data:")
            
            # Display the aspect data
            for key, value in aspect.data.items():
                print(f"      {key}: {value}")
            print()
            
            found_aspects += 1
            
        except Exception as aspect_error:
            print(f"⚠️  Aspect '{aspect_id}' not found: {str(aspect_error)[:80]}")
            print()
    
    if found_aspects == len(aspect_ids):
        print()
        print(f"✅ Validation passed: All {found_aspects} aspects are readable and properly formatted!")
    else:
        print()
        print(f"⚠️  Partial validation: Found {found_aspects}/{len(aspect_ids)} aspects")
    
except Exception as e:
    print(f"❌ Validation failed: {e}")
    print()
    print("This might indicate:")
    print("   - Entry doesn't exist in the catalog yet")
    print("   - Aspects weren't successfully created")
    print("   - Permissions issue reading from Dataplex Catalog")
    raise

In [ ]:
# Section 4 completion and next steps
print()
print("🔗 View in Google Cloud Console:")
print()
print(f"   Dataplex Catalog Search:")
print(f"   https://console.cloud.google.com/dataplex/search?project={PROJECT_ID}")
print()
print(f"   Sample Table Entry (blockgroup_2018_5yr):")
print(f"   https://console.cloud.google.com/dataplex/search;entry={PROJECT_ID}.{DATASET_ID}.blockgroup_2018_5yr?project={PROJECT_ID}")
print()
print(f"   BigQuery Dataset:")
print(f"   https://console.cloud.google.com/bigquery?project={PROJECT_ID}&d={DATASET_ID}&p={PROJECT_ID}&page=dataset")
print()
print("=" * 60)
print("✅ Section 4 complete! All census tables now have rich metadata!")
print("=" * 60)
print()
print("📊 What you've accomplished:")
print("   ✅ Copied 278 census tables to your project")
print("   ✅ Created custom aspect types for survey and governance metadata")
print("   ✅ Applied aspects to all tables with parsed metadata")
print("   ✅ Enriched your data catalog with searchable, structured metadata")
print()
print("🔍 Try searching in Dataplex Catalog:")
print("   - Search for 'survey_vintage:2018' to find 2018 tables")
print("   - Search for 'estimate_period:5_year' to find 5-year estimates")
print("   - Search for 'source_agency:Census' to find census data")
print()
print("📝 Next Steps:")
print("   - Explore the Dataplex Catalog UI to see your metadata")
print("   - Query BigQuery tables with context from the metadata")
print("   - Create additional aspect types for your own datasets")
print("   - Use Dataplex Data Quality to profile and validate data")

In [ ]:
# Console links and next steps
print()
print("🔗 View in Google Cloud Console:")
print(f"   All Aspect Types: https://console.cloud.google.com/dataplex/govern/aspect-types?project={PROJECT_ID}")
print(f"   Census Survey Metadata: https://console.cloud.google.com/dataplex/govern/aspect-types/{aspect_type_id}?project={PROJECT_ID}")
print(f"   Public Data Governance: https://console.cloud.google.com/dataplex/govern/aspect-types/{governance_aspect_type_id}?project={PROJECT_ID}")
print()
print("✅ Section 3 complete! Both aspect types are ready:")
print("   1. Census Survey Metadata - for survey vintage and estimate period")
print("   2. Public Data Governance - for source agency and licensing info")
print()
print("📝 Next Steps:")
print("   - These aspect types can now be attached to BigQuery table entries")
print("   - Use the Dataplex Catalog API to apply aspects with metadata values")
print("   - Example: Attach survey_vintage=2018, estimate_period='5-Year Estimate' to blockgroup_2018_5yr")
print("   - Example: Attach source_agency='US Census Bureau', data_license='Public Domain' for governance")